# Lektion 18 (opfølgning): Kvitteringer der Beviser, at en *Menneske* Godkendte Handlingen

Lektionen beviser, hvad **agenten** gjorde, og hvad **porten** besluttede. Denne notesbog tilføjer den manglende halvdel: bevis for at et **navngivet menneske** godkendte den **præcise** handling — en separat, menneskelig underskrift over den fulde kanoniske handling, verificeret offline.

Begge artefakter her bruger **den samme konvolutform som lektionens kvitteringer**: en flad nyttelast med et `type`-felt, underskrevet med Ed25519 over SHA-256 af de kanoniske nyttelast-byte, med et struktureret `signature`-objekt vedhæftet (og udelukket fra de underskrevne byte). Godkendelses-kvitteringen er en ny `type` (`human.approval.v1`) ved siden af handlingstypen, så en enkelt `verify_chain` dækker begge artefaktyper med den samme kodevej, du byggede i hovednotebooken. Dette er også formen på medunderskriftsstien i Internet-Draften, som lektionen følger (draft-farley-acta-signed-receipts).

En bevidst opgradering i forhold til demo-verifikatoren i hovednotebooken: verifikatoren her løser `signature.key_id` mod et **fastlåst nøgleregister** i stedet for at stole på en offentlig nøgle, der bæres inde i kvitteringen. Det er den produktionsholdning, som lektionens egen tjekliste anbefaler ("offentliggør verifikationsnøglen"), og det er hvad der gør forfalskning til et afvisning i stedet for en medbringer-selv-nøgle-omgåelse.

Reglen denne notesbog lærer: **en signeret godkendelse er ikke myndighed i sig selv.** Myndighed eksisterer kun, hvis godkendelses-kvitteringen og handlingskvitteringen stadig binder til den samme kanoniske handling ved udførelsestidspunktet, under en politikversion, nøgle og udløb, der stadig er aktuelle, og en godkendelse der ikke allerede er blevet brugt. Hver fejl afvises med en **distinkt årsag**, så du kan skelne mellem *myndighed blev forældet* og *den udførte handling ændrede sig*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Den præcise handling

Enheden for godkendelse er **det kanoniske handlingsobjekt** — ikke et upræcist mærke som "godkend refundering," men den nøjagtige, fuldt specificerede handling. At underskrive hele objektet (og udlede en digest derfra) er det, der senere lader os bevise, at mennesket godkendte *dette* og intet andet.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## En kuvert, to myndigheder

Hver kvittering er lektionens kuvert: en flad nyttelast med et `type`-felt plus et `signature`-objekt (`alg`, `sig`, `key_id`), som **ikke** er en del af de signerede bytes. `verify_envelope` er den fælles strukturelle + signaturkontrol for begge kvitteringstyperne; hvilken **pinned key registry** den løser `signature.key_id` imod, er hvad der holder myndighederne adskilte:

- **godkendelseskvittering** (`human.approval.v1`) – navngivet godkender, den fulde kanoniske handling **og dens digest**, `policy_version`, udstedelses- og udløbstidsstempler. Engangsforsyning spores på kædeniveau.
- **handlingskvittering** (`agent.action.v1`) – agentidentitet, `run_id`, den samme kanoniske handlings **digest**, udførelsesresultat + tidsstempel, og `parent_approval_ref`: `receipt_hash` for godkendelsen, samme konvention som `previous_receipt_hash` i lektionens kæde.

Det fælles `action_digest`-felt er det sammenbindende, bindingen afhænger af. `key_id` findes i signaturobjektet kun som et opslagstips: at pege det på en anden pinned key får signaturkontrollen til at fejle, så det giver intet.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: hvor bindningen faktisk besluttes

`verify_chain` er **ikke** en bekvemmelighedsomslag over to signaturkontroller. Det er det eneste sted, hvor den delte kanoniske `action_digest`, politikken/nøglen/udløbsdatoens **aktualitet** for godkendelsen, og den **engangsforbrug** af godkendelsen tjekkes samlet, mod den handling, der udføres *lige nu*.

Hver fejlfinding afvises med en **forskellig grund**, så en læser af afvisningen kan se, om myndigheden blev forældet (politik flyttet, nøgle roteret, godkendelse udløbet, godkendelse forbrugt) eller den udførte handling ændrede sig under en stadig gyldig godkendelse (digest-udskiftning).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Hvad bindingen fanger

Hver sag nedenfor fejler **lukket** med en **distinkt grund**. Det første blok er det klassiske sæt (manipulation, forvirret stedfortræder, genafspilning, forfalskning på enhver myndighed, forkert formateret input). Det andet blok er det par, der gør egenskaben reel fremfor blot påstået:

- **ældet myndighed** — signaturen er stadig gyldig, men politikversionen er flyttet, godkendernøglen blev roteret ud af den fastlåste registry, eller godkendelsen udløb før udførelsen;
- **digest-udskiftning** — en gyldigt signeret handlingskvittering, hvis `parent_approval_ref` peger på en *reel* godkendelse, men den godkendelses kanoniske handlingsdigest matcher ikke den handling, der faktisk udføres.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Hvad dette beviser — og hvad det ikke gør

**Beviser:** en navngiven person godkendte *præcis denne kanoniske handling* (fuld handling + digest, underskrevet med en nøgle hentet fra et fastgjort register), og agenten udførte *præcis den godkendte handling* (samme digest, kvittering bundet til godkendelsen ved `receipt_hash`, lektionens egen kæde-konvention) — mens godkendelsens politikversion, nøgle og udløbstid stadig var gyldige, præcis én gang. Hvis en af siderne ændres, fejler kæden lukket, og afvisningsårsagen fortæller dig **hvilken** egenskab der brød: forældet autoritet vs. en ændret handling.

**Beviser ikke:** at godkendelses-UI’en viste personen det, de troede, de underskrev (WYSIWYS er sit eget problem), at nøglen ikke blev tvunget eller stjålet før rotation, eller at efterfølgende virkninger matchede handlingen. Underskrevet ≠ autoriseret: en gyldig signatur på en forældet politik, en roteret nøgle, et udløbet vindue eller en anden digest giver intet her.

De to typer kvitteringer deler lektionens kuvert og en `verify_chain` kodevej med vilje: den binding, du byggede til handlingens kvitteringer i hovednotebooken, er den samme kode, der tjekker personens godkendelse. En verifier-kontrakt, separate fastgjorte myndigheder, forenet af den kanoniske handlings digest og intet andet.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
